# Lab 3.1 – Muster finden: drei Fragen an die Nordwind-Daten

**Beispiel (fertige Lösung).** Zellen der Reihe nach mit **Umschalt+Enter** ausführen.

| Frage | Werkzeug | Ergebnis |
| --- | --- | --- |
| 1 · Wie wächst jede Region? | `pivot_table` + `pct_change` | `wachstum` |
| 2 · Welche Produktgruppe verdient am meisten je Euro Umsatz? | `groupby` + Summen | `marge` |
| 3 · Wie entwickeln sich die Kanäle Monat für Monat? | `pivot_table` | `kanal_monat`  |

Eingabe: `umsatz_features.csv` aus dem Beispiel von Lab 2.3. Die drei Tabellen gehen als CSV nach `project/examples/m03-exploration/l01-eda/`.

In [1]:
import pandas as pd

EINGABE = "project/examples/m02-data-preparation/l03-features/umsatz_features.csv"
ORDNER = "project/examples/m03-exploration/l01-eda/"     # hier landen die drei Ergebnistabellen

df = pd.read_csv(EINGABE, parse_dates=["datum"])   # Spalte datum gleich als Datum lesen
print(len(df), "Zeilen von", df["monat"].min(), "bis", df["monat"].max())
df.head(3)

33156 Zeilen von 2024-01 bis 2026-09


,beleg_nr,datum,jahr,kunde_id,region,produktgruppe,kanal,menge,umsatz,kosten,kosten_fehlt,monat,quartal,deckungsbeitrag,marge
0,B000001,2024-01-01,2024,K0362,Nord,Schreibwaren,Onlineshop,13,39.89,25.92,False,2024-01,2024Q1,13.97,0.350213
1,B000002,2024-01-01,2024,K0281,Nord,Schreibwaren,Onlineshop,37,112.90,80.85,False,2024-01,2024Q1,32.05,0.283880
2,B000003,2024-01-01,2024,K0341,Nord,Technik,Außendienst,39,976.83,795.65,False,2024-01,2024Q1,181.18,0.185478


## Überblick: Wie sind die Werte verteilt?

`describe()` zeigt je Zahlenspalte Anzahl, Mittelwert (`mean`), Streuung (`std`), Minimum, Viertelwerte und Maximum.

In [2]:
df[["menge", "umsatz", "marge"]].describe().round(2)

,menge,umsatz,marge
count,33156.00,33156.00,32508.00
mean,35.42,295.56,0.25
std,18.34,331.23,0.08
min,2.00,6.35,0.03
25%,23.00,131.76,0.19
50%,32.00,215.33,0.26
75%,44.00,349.64,0.31
max,186.00,32800.00,0.47


## Frage 1: Wie wächst jede Region?

Verglichen werden je Jahr nur die Monate Januar bis September, weil 2026 erst bis September vorliegt.
`pct_change(axis="columns")` rechnet die Veränderung von Spalte zu Spalte, also Jahr gegen Vorjahr.

In [3]:
def wachstum_je_region(df, bis_monat=9):
    """Frage 1: Umsatz Januar bis bis_monat je Region und Jahr, dazu Veränderung zum Vorjahr in %."""
    zeitraum = df[df["datum"].dt.month <= bis_monat]              # gleiche Monate in jedem Jahr
    tabelle = zeitraum.pivot_table(index="region", columns="jahr", values="umsatz", aggfunc="sum")
    letztes_jahr = tabelle.columns.max()
    veraenderung = tabelle.pct_change(axis="columns") * 100       # Jahr gegen Vorjahr in %
    tabelle["wachstum_prozent"] = veraenderung[letztes_jahr]
    tabelle.columns = [str(spalte) for spalte in tabelle.columns]  # 2024 -> "2024"
    return tabelle.round(1).sort_values("wachstum_prozent", ascending=False)


wachstum = wachstum_je_region(df)
wachstum

,2024,2025,2026,wachstum_prozent
region,,,,
West,656796.9,721872.8,792757.2,9.8
Süd,594101.8,617795.6,674400.2,9.2
Ost,513457.4,477187.9,520396.9,9.1
Nord,750940.8,792446.5,826874.9,4.3


## Frage 2: Welche Produktgruppe verdient am meisten je Euro Umsatz?

Marge einer Gruppe = Summe Deckungsbeitrag / Summe Umsatz (Regel aus Lab 2.3), nie der Mittelwert der Zeilenmargen.

In [4]:
def marge_je_produktgruppe(df):
    """Frage 2: Marge je Produktgruppe = Summe Deckungsbeitrag / Summe Umsatz."""
    summen = df.groupby("produktgruppe")[["umsatz", "deckungsbeitrag"]].sum()
    summen["marge_prozent"] = summen["deckungsbeitrag"] / summen["umsatz"] * 100
    return summen.round(1).sort_values("marge_prozent", ascending=False)


marge = marge_je_produktgruppe(df)
marge

,umsatz,deckungsbeitrag,marge_prozent
produktgruppe,,,
Schreibwaren,747713.0,260383.0,34.8
Ordnung,1165892.9,337800.7,29.0
Küche,909567.4,246211.4,27.1
Reinigung,1342746.3,323431.6,24.1
Papier,2818190.6,481976.9,17.1
Technik,2815639.6,399081.1,14.2


## Frage 3: Wie entwickeln sich die Kanäle Monat für Monat?

Gesucht ist eine Tabelle `kanal_monat`: eine Zeile je Monat, je eine Spalte Umsatz für `Außendienst` und `Onlineshop`, dazu `anteil_online_prozent`.

In [5]:
# >>> AUFGABE 1: Tabelle kanal_monat erzeugen – Prompt siehe Übung, Aufgabe 3
def kanal_je_monat(df):
    """Frage 3: Umsatz je Monat und Kanal, dazu der Anteil des Onlineshops in %."""
    tabelle = df.pivot_table(index="monat", columns="kanal", values="umsatz", aggfunc="sum")
    tabelle["anteil_online_prozent"] = tabelle["Onlineshop"] / (tabelle["Außendienst"] + tabelle["Onlineshop"]) * 100
    tabelle.columns.name = None                                    # Überschrift "kanal" entfernen
    return tabelle.round(1)


kanal_monat = kanal_je_monat(df)
# <<< ENDE AUFGABE 1
kanal_monat.tail()

,Außendienst,Onlineshop,anteil_online_prozent
monat,,,
2026-05,221267.0,82293.7,27.1
2026-06,220478.1,74985.2,25.4
2026-07,196548.9,96508.2,32.9
2026-08,230718.3,99070.7,30.0
2026-09,240058.1,103323.6,30.1


In [6]:
# Prüfzelle – nicht ändern
def pruefe_kanal_monat(t):
    if not isinstance(t, pd.DataFrame):
        return "nicht OK – kanal_monat ist noch keine Tabelle (DataFrame)."
    fehlt = {"Außendienst", "Onlineshop", "anteil_online_prozent"} - set(t.columns)
    if fehlt:
        return "nicht OK – Spalten fehlen: " + ", ".join(sorted(fehlt)) + ". Kanäle als Spalten: pivot_table mit columns=\"kanal\"."
    if len(t) != 33:
        return f"nicht OK – {len(t)} Zeilen statt 33 (eine Zeile je Monat, 2024-01 bis 2026-09)."
    anteil = float(t["anteil_online_prozent"].iloc[-1])
    if abs(anteil - 30.1) > 0.1:
        return f"nicht OK – Onlineshop-Anteil im letzten Monat {anteil:.1f} statt 30.1: Onlineshop / (Außendienst + Onlineshop) * 100."
    return "OK – 33 Monate, Onlineshop-Anteil im September 2026: 30,1 %"


print(pruefe_kanal_monat(kanal_monat))

OK – 33 Monate, Onlineshop-Anteil im September 2026: 30,1 %


## Ergebnisse speichern

Die drei Tabellen gehen als CSV in den Ordner `ORDNER`. Lab 3.2 liest dieselben Dateien aus dem Beispielordner.

In [7]:
wachstum.to_csv(ORDNER + "wachstum_region.csv")
marge.to_csv(ORDNER + "marge_produktgruppe.csv")
if isinstance(kanal_monat, pd.DataFrame):
    kanal_monat.to_csv(ORDNER + "kanal_monat.csv")
    print("Gespeichert: drei Tabellen in", ORDNER)
else:
    print("Gespeichert: zwei Tabellen in", ORDNER, "– kanal_monat fehlt noch (Aufgabe 1)")

Gespeichert: drei Tabellen in project/examples/m03-exploration/l01-eda/


## Erweiterung (Beispiel): Steigt der Umsatz, weil mehr Kunden kaufen oder weil sie mehr kaufen?

Eine weitere Analysefrage, beantwortet mit denselben Werkzeugen: Anzahl Kunden (`nunique`) und Umsatz je Kunde, Januar bis September je Jahr.

In [8]:
jan_sep = df[df["datum"].dt.month <= 9]
kunden = jan_sep.groupby("jahr").agg(kunden=("kunde_id", "nunique"), umsatz=("umsatz", "sum"))
kunden["umsatz_je_kunde"] = kunden["umsatz"] / kunden["kunden"]
kunden["veraenderung_umsatz_je_kunde_prozent"] = kunden["umsatz_je_kunde"].pct_change() * 100
kunden.round(1)

,kunden,umsatz,umsatz_je_kunde,veraenderung_umsatz_je_kunde_prozent
jahr,,,,
2024,395,2515296.9,6367.8,NaN
2025,398,2609302.9,6556.0,3.0
2026,367,2814429.1,7668.7,17.0
